# Télos: Lightning AI TPU v6e Training Notebook

This notebook orchestrates the training of **COROSRED** across 15M, 50M, and 100M models on a **Lightning AI TPU v6e-1 (Trillium)**.

> ⚠️ **CRITICAL TPU RUNTIME RULE**:
> Never call `xm.xla_device()` inside interactive notebook kernel cells. On Linux, `/dev/vfio/0` can only be locked by a single process at a time. If the interactive Jupyter kernel binds the TPU, shell subprocesses (`!python ...`) will crash with `open(/dev/vfio/0): Device or resource busy`.
>
> All training and hardware verification in this notebook is executed via clean subprocesses.

In [ ]:
# Cell 1: Environment Diagnostics & TPU Verification (Non-Locking)
import os
import sys
from pathlib import Path

# Automatically resolve and switch to telos_repo directory
cand_roots = [
    Path("/teamspace/studios/this_studio/telos_repo"),
    Path.cwd(),
    Path.cwd().parent,
]
project_root = None
for cand in cand_roots:
    if (cand / "scripts" / "train.py").exists() and (cand / "telos").exists():
        project_root = cand.resolve()
        break

if project_root is None:
    project_root = Path.cwd()

os.chdir(project_root)
sys.path.insert(0, str(project_root))
print(f"✓ Working directory set to: {project_root}")

# Clear any stale device locks
!fuser -k /dev/vfio/* 2>/dev/null || true

# Test TPU in a clean subprocess (avoids locking the Jupyter kernel)
!PJRT_DEVICE=TPU python -c "import torch_xla.core.xla_model as xm; dev = xm.xla_device(); print(f'✓ TPU Subprocess Active: {dev} ({xm.xla_device_hw(dev)})')"

In [ ]:
# Cell 2: Download 15M & 50M Phase A Checkpoints from Hugging Face
import os
import shutil
from huggingface_hub import hf_hub_download

print("=" * 70)
print("Downloading 15M & 50M Phase A Checkpoints...")
print("=" * 70)

# 1. 15M Phase A
os.makedirs("checkpoints/corosred/15m/phase_a", exist_ok=True)
p15 = hf_hub_download(repo_id="Kazenowoko/telos-corosred-15m-ar", filename="checkpoint_final.pt")
shutil.copy(p15, "checkpoints/corosred/15m/phase_a/checkpoint_final.pt")
print("✓ 15M Phase A checkpoint ready at checkpoints/corosred/15m/phase_a/checkpoint_final.pt")

# 2. 50M Phase A
os.makedirs("checkpoints/corosred/50m/phase_a", exist_ok=True)
p50 = hf_hub_download(repo_id="Kazenowoko/telos-corosred-50m-ar", filename="checkpoint_final.pt")
shutil.copy(p50, "checkpoints/corosred/50m/phase_a/checkpoint_final.pt")
print("✓ 50M Phase A checkpoint ready at checkpoints/corosred/50m/phase_a/checkpoint_final.pt")

## Training Execution
Each run resets device locks with `fuser -k /dev/vfio/*` and launches `scripts/train.py` with `PJRT_DEVICE=TPU`.

In [ ]:
# Cell 3: Train 15M COROSRED Phase B (Confidence-Routed Selective Re-Diffusion)
DATA_PATH = "data/corpus.bin"  # Or use --synthetic if corpus is not yet generated

!fuser -k /dev/vfio/* 2>/dev/null || true
!PJRT_DEVICE=TPU python scripts/train.py \
  --paradigm corosred \
  --phase B \
  --params 15m \
  --hardware xla \
  --batch-size 384 \
  --grad-accum 1 \
  --devices 1 \
  --init-checkpoint checkpoints/corosred/15m/phase_a/checkpoint_final.pt \
  --checkpoint-dir checkpoints/corosred/15m/phase_b \
  --self-condition \
  --self-cond-prob 0.5 \
  --data {DATA_PATH}

In [ ]:
# Cell 4: Train 50M COROSRED Phase B (Confidence-Routed Selective Re-Diffusion)
DATA_PATH = "data/corpus.bin"

!fuser -k /dev/vfio/* 2>/dev/null || true
!PJRT_DEVICE=TPU python scripts/train.py \
  --paradigm corosred \
  --phase B \
  --params 50m \
  --hardware xla \
  --batch-size 384 \
  --grad-accum 1 \
  --devices 1 \
  --init-checkpoint checkpoints/corosred/50m/phase_a/checkpoint_final.pt \
  --checkpoint-dir checkpoints/corosred/50m/phase_b \
  --self-condition \
  --self-cond-prob 0.5 \
  --data {DATA_PATH}

In [ ]:
# Cell 5: Train 100M COROSRED Phase A (Causal AR + Learned Reliability Head)
DATA_PATH = "data/corpus.bin"

!fuser -k /dev/vfio/* 2>/dev/null || true
!PJRT_DEVICE=TPU python scripts/train.py \
  --paradigm corosred \
  --phase A \
  --params 100m \
  --hardware xla \
  --batch-size 384 \
  --grad-accum 1 \
  --devices 1 \
  --checkpoint-dir checkpoints/corosred/100m/phase_a \
  --data {DATA_PATH}

In [ ]:
# Cell 6: Train 100M COROSRED Phase B (Confidence-Routed Selective Re-Diffusion)
DATA_PATH = "data/corpus.bin"

!fuser -k /dev/vfio/* 2>/dev/null || true
!PJRT_DEVICE=TPU python scripts/train.py \
  --paradigm corosred \
  --phase B \
  --params 100m \
  --hardware xla \
  --batch-size 384 \
  --grad-accum 1 \
  --devices 1 \
  --init-checkpoint checkpoints/corosred/100m/phase_a/checkpoint_final.pt \
  --checkpoint-dir checkpoints/corosred/100m/phase_b \
  --self-condition \
  --self-cond-prob 0.5 \
  --data {DATA_PATH}

## Checkpoint Audit
Inspect the trained checkpoint files and parameter tensors.

In [ ]:
# Cell 7: Inspect Checkpoints
from pathlib import Path
import torch

checkpoints = [
    "checkpoints/corosred/15m/phase_b/checkpoint_final.pt",
    "checkpoints/corosred/50m/phase_b/checkpoint_final.pt",
    "checkpoints/corosred/100m/phase_a/checkpoint_final.pt",
    "checkpoints/corosred/100m/phase_b/checkpoint_final.pt",
]

print("=" * 70)
print("CHECKPOINT AUDIT:")
print("=" * 70)
for path in checkpoints:
    p = Path(path)
    if p.exists():
        size_mb = p.stat().st_size / (1024 * 1024)
        sd = torch.load(p, map_location="cpu", weights_only=False)
        keys = sd.get("model_state_dict", sd).keys()
        print(f"✓ {path:55s} | {size_mb:6.1f} MB | {len(keys)} params")
    else:
        print(f"✗ Not found: {path}")